# Excluded cases report

This notebook exports the excluded judgments with the remarks recorded in the verification workflow.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (repo_root / 'featureExtraction' / '.env', repo_root / 'featureVerification' / '.env.local', repo_root / '.env'):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import get_collection

verified_collection, _ = get_collection()
query = {'is_verified': True, 'exclude': True}
projection = {'filename': 1, 'judgement.neutral_citation': 1, 'remarks': 1, 'exclude_reason': 1, 'trials': 1}
docs = list(verified_collection.find(query, projection))

def normalize_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, list):
        return '\n'.join(str(item) for item in value if item is not None)
    return str(value)

rows = []
for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    rows.append({
        'neutral_citation': (doc.get('judgement') or {}).get('neutral_citation'),
        'remarks': normalize_text(doc.get('remarks')),
        'trial_count': len(trials),
    })

df = pd.DataFrame(rows)
output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    df.to_excel(output_dir / 'excluded_cases_report.xlsx', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Saved {len(df)} excluded cases to {output_dir / "excluded_cases_report.xlsx"}')
df.head()

/Users/cxiang/Projects/drug-trafficing-sentence-predictor/featureExtraction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 172 excluded cases to /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/excluded_cases_report.xlsx


,neutral_citation,remarks,trial_count
0,[2021] HKCFI 2372,The judge adopted a global approach for the ov...,2
1,[2021] HKDC 1500,the judge adopted a global approach for the ov...,2
2,[2021] HKDC 1503,Defendant was NOT convicted of drug traffickin...,1
3,[2021] HKCFI 2607,The judge did not consider the sentence for bo...,2
4,[2021] HKCFI 1269,The judge adopted a global approach in determi...,2
